LIBRARIES LIKE PYPDF, TRANSFORMERS, SENTENCE TRANSFORMERS, LANGCHAIN LIKE TO PRINT MESSAGES LIKE WARNING:INFO:FUTURE WARNING. THESE CAN CLUTTER THE NOTEBOOK. THIS CELL CLEANS UP THE OUTPUT SO WE CAN FOCUS ON ONLY THE RESULTS.

    

In [ ]:
import logging                                           
import os                                                  
import warnings                                             

warnings.filterwarnings("ignore")                           

logging.getLogger().setLevel(logging.CRITICAL)              
logging.getLogger("pypdf").setLevel(logging.ERROR)      

os.environ["TRANSFORMERS_VERBOSITY"] = "error"            

THIS CELL INSTALLS AAL OF THE TOOLS THAT WE'LL NEED. [IT HELPS US LOAD OUR GRA DOCUMENTS --> SPLIT DTHE DOCUMENTS --> CREATE EMBEDDINGS --> BULD FAISS INDEX(VECTOR_DATABASE) --> FINALLY ANSWER TEXT QUESTIONS]

In [ ]:
import sys                                      
!{sys.executable} -m pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface huggingface_hub faiss-cpu    
                    

In [ ]:
import os
import sys

# 1. Install required multi-format document parsers securely
!"{sys.executable}" -m pip install -q pypdf tqdm sentence-transformers docx2txt      

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, DirectoryLoader                
from langchain_text_splitters import RecursiveCharacterTextSplitter                                          
from langchain_community.embeddings import HuggingFaceEmbeddings                                                
from langchain_community.vectorstores import FAISS                                                          

all_documents = []

 # 1. DATA INGESTION

# --- A. SCAN AND LOAD ALL PDF FILES ---
if os.path.exists('GRA data'):                                                                              
    pdf_loader = DirectoryLoader('GRA data', glob="./*.pdf", loader_cls=PyPDFLoader)                        
    pdf_docs = pdf_loader.load()                                                                            
    all_documents.extend(pdf_docs)                                                                         

# --- B. SCAN AND LOAD ALL WORD FILES ---
    word_loader = DirectoryLoader('GRA data', glob="./*.docx", loader_cls=Docx2txtLoader)
    word_docs = word_loader.load()
    all_documents.extend(word_docs)
    print(f"Successfully loaded {len(word_docs)} Word documents.")

print(f"Total raw document records collected: {len(all_documents)}")

In [ ]:
pip install matplotlib wordcloud

Plot 1: Document Page Count Comparison

In [ ]:
import os
import matplotlib.pyplot as plt
from langchain_community.document_loaders import PyPDFLoader

# 1. Point to your data folder where all the PDFs live
data_folder = "./GRA data"  
doc_names = []
page_counts = []

# 2. Automatically loop through the folder and count pages
for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(data_folder, file_name)
        try:
            # Quickly load the PDF to count its pages
            loader = PyPDFLoader(file_path)
            pages = loader.load()
            
            # Save the file name and the total page count
            doc_names.append(file_name)
            page_counts.append(len(pages))
        except Exception as e:
            print(f"Could not read {file_name}: {e}")

# 3. Create the bar chart automatically
plt.figure(figsize=(12, 6))
# If you have a LOT of files, a horizontal bar chart (barh) is much easier to read!
plt.barh(doc_names, page_counts, color='#4682b4', edgecolor='black', height=0.6)

# 4. Add professional styling
plt.title("Automated Data Scale Analysis: Total Pages per Document", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Pages", fontsize=12)
plt.ylabel("Source PDF Files", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 5. Put the numbers next to the bars
for i, v in enumerate(page_counts):
    plt.text(v + 1, i, str(v), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

Plot 2: Total Word Count per Document

In [ ]:
import os
import matplotlib.pyplot as plt
from langchain_community.document_loaders import PyPDFLoader

# 1. Automatically scan the data folder
data_folder = "./GRA data"  
doc_names = []
word_counts = []

for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(data_folder, file_name)
        try:
            # Load the document
            loader = PyPDFLoader(file_path)
            pages = loader.load()
            
            # Extract all text across all pages and count the words
            total_text = "".join([page.page_content for page in pages])
            words = total_text.split()
            
            doc_names.append(file_name)
            word_counts.append(len(words)) # Saves the exact, real word count!
        except Exception as e:
            print(f"Could not read {file_name}: {e}")

# 2. Plot the real word counts automatically
plt.figure(figsize=(12, 6))
plt.barh(doc_names, word_counts, color='#e67e22', edgecolor='black', height=0.6)

# 3. Add professional styling
plt.title("Automated Text Volume Analysis: Real Word Count per Document", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Words", fontsize=12)
plt.ylabel("Source PDF Files", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 4. Display the exact numbers next to the bars
for i, v in enumerate(word_counts):
    plt.text(v + 500, i, f"{v:,} words", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
pip install pdfplumber

In [ ]:
import os
import matplotlib.pyplot as plt
import pdfplumber

# 1. Automatically scan the data folder
data_folder = "./GRA data"  
doc_names = []
word_counts = []

for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(data_folder, file_name)
        try:
            total_words = 0
            # Open the PDF with pdfplumber for deep extraction
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text = page.extract_text()
                    if text:
                        total_words += len(text.split())
            
            doc_names.append(file_name)
            word_counts.append(total_words)
        except Exception as e:
            print(f"Could not read {file_name}: {e}")

# 2. Plot the real word counts automatically
plt.figure(figsize=(12, 6))
plt.barh(doc_names, word_counts, color='#e67e22', edgecolor='black', height=0.6)

# 3. Add professional styling
plt.title("Automated Text Volume Analysis: Real Word Count per Document", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Words", fontsize=12)
plt.ylabel("Source PDF Files", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# 4. Display the exact numbers next to the bars
for i, v in enumerate(word_counts):
    plt.text(v + 500, i, f"{v:,} words", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

Plot 3: Document Word Cloud (The Presenter's Favorite)

In [ ]:
import os
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from langchain_community.document_loaders import PyPDFLoader

# 1. Automatically scan the data folder
data_folder = "./GRA data"  
all_text_list = []

for file_name in os.listdir(data_folder):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(data_folder, file_name)
        try:
            loader = PyPDFLoader(file_path)
            pages = loader.load()
            
            # Extract text from each page and save it
            for page in pages:
                all_text_list.append(page.page_content)
        except Exception as e:
            print(f"Could not read {file_name}: {e}")

# 2. Combine ALL extracted pages into one massive string of text
real_combined_text = " ".join(all_text_list)

# 3. Generate the Word Cloud using your REAL data
wordcloud = WordCloud(
    width=800, 
    height=400, 
    background_color='white', 
    colormap='viridis',      
    max_words=100,           
    contour_width=1, 
    contour_color='steelblue'
).generate(real_combined_text)

# 4. Display the beautiful image
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")  
plt.title("Semantic Domain Analysis: High-Frequency Keywords", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
pip install nltk

In [ ]:
import collections
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords

# Download the official dictionary of stopwords
nltk.download('stopwords')

# Load the official English stopwords into a set
stop_words_set = set(stopwords.words('english'))


# 2. Process your real text strings (using the real_combined_text from previous step)
# Clean punctuation and split into individual lowercase words
clean_text = real_combined_text.lower().replace('.', ' ').replace(',', ' ').replace(';', ' ')
all_words = [word for word in clean_text.split() if word.isalnum()]

# 3. Categorize words into Stopwords vs Content Words
stopwords_found = [word for word in all_words if word in stop_words_set]
content_words_found = [word for word in all_words if word not in stop_words_set]

# --- PLOT 1: PIE CHART (RATIO) ---
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
labels = ['Stopwords (Filler Words)', 'Content Words (Meaningful)']
sizes = [len(stopwords_found), len(content_words_found)]
colors = ['#e74c3c', '#2ecc71']
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=colors, 
        textprops={'fontweight':'bold'})
plt.title("Text Composition: Stopwords vs. Content Words", fontsize=12, fontweight='bold', pad=15)

# --- PLOT 2: BAR CHART (TOP 10 STOPWORDS) ---
plt.subplot(1, 2, 2)
stopword_counts = collections.Counter(stopwords_found)
top_10_stopwords = stopword_counts.most_common(10)

words, counts = zip(*top_10_stopwords)
plt.bar(words, counts, color='#34495e', edgecolor='black', width=0.6)
plt.title("Top 10 Most Frequent Stopwords", fontsize=12, fontweight='bold', pad=15)
plt.ylabel("Frequency (Count)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Display counts on top of bars
for i, v in enumerate(counts):
    plt.text(i, v + 2, str(v), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. Let's simulate sentence length data based on typical legal text structures
# In your real pipeline, you could use text.split('.') to measure actual sentence lengths
sample_sentence_lengths = [12, 18, 45, 62, 34, 55, 72, 28, 41, 53, 89, 14, 32, 61, 48]

plt.figure(figsize=(9, 4))
plt.hist(sample_sentence_lengths, bins=5, color='#2ecc71', edgecolor='black', alpha=0.8)

# Add styling and labels
plt.title("Structural Complexity: Sentence Length Distribution", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Words in a Single Sentence", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)
plt.axvline(30, color='red', linestyle='dashed', linewidth=2, label='Human Readability Limit (~30 words)')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

DOCUMENT PROCESSING PIPELINE:
    1. DATA INGESTION
    2. PREPROCESSING
    3. VECTORIZATION PIPELINE



In [ ]:
import os
import sys

#  Install required multi-format document parsers securely
!"{sys.executable}" -m pip install -q pypdf tqdm sentence-transformers docx2txt 

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, DirectoryLoader         
from langchain_text_splitters import RecursiveCharacterTextSplitter                                          
from langchain_community.embeddings import HuggingFaceEmbeddings                                         
from langchain_community.vectorstores import FAISS                                                          

all_documents = []
data_dir = 'GRA data'

# =====================================================================
# 1. DATA INGESTION
# =====================================================================

if os.path.exists(data_dir):
    # --- A. SCAN AND LOAD ALL PDF FILES ---
    print(f"Scanning '{data_dir}' for PDF guidelines...")
    pdf_loader = DirectoryLoader(data_dir, glob="./*.pdf", loader_cls=PyPDFLoader)                        
    pdf_docs = pdf_loader.load()                                                                           
    all_documents.extend(pdf_docs)                                                                         
    print(f"Successfully loaded {len(pdf_docs)} PDF documents.")

    # --- B. SCAN AND LOAD ALL WORD FILES ---
    print(f"Scanning '{data_dir}' for Word documentation...")
    word_loader = DirectoryLoader(data_dir, glob="./*.docx", loader_cls=Docx2txtLoader)
    word_docs = word_loader.load()
    all_documents.extend(word_docs)
    print(f"Successfully loaded {len(word_docs)} Word documents.")
else:
    print(f"✗ Error: The directory '{data_dir}' does not exist. Please create it and drop your files inside.")
    sys.exit(1)

print(f"\nTotal raw document records collected: {len(all_documents)}")


# =====================================================================
# 2. TEXT PREPROCESSING, SEMANTIC CHUNKING, AND METADATA CLEANING
# =====================================================================

# Split both formats using unified chunk sizes (Tuned to Guidebook specs)        
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=75)                      
text_chunks = text_splitter.split_documents(all_documents)                                              

# Clean up metadata paths to extract clean file names for accurate citations
for chunk in text_chunks:                                                                
    if "source" in chunk.metadata:                                                                      
        # Converts full system path (e.g., 'GRA data/Penalty and Interest Waiver Act 2021 (Act 1065).pdf')
        # into a pristine citation name: 'Penalty and Interest Waiver Act 2021 (Act 1065).pdf'
        chunk.metadata["filename"] = os.path.basename(chunk.metadata["source"])                         

print(f"Split everything into {len(text_chunks)} clean text chunks with source metadata.")


# =====================================================================
# 3. CREATING THE VECTOR EMBEDDINGS AND LOCAL INDEX STORAGE
# =====================================================================

print("\nInitializing text embedding matrix (sentence-transformers/all-MiniLM-L6-v2)...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")                 

# 4. Build and save your updated local vector index
print("Encoding tokens and building local FAISS vector store indexes...")
vector_store = FAISS.from_documents(text_chunks, embeddings)                                          
vector_store.save_local("faiss_index")                                                                

print("\n🌟 Success! Your vector database has been rebuilt with all PDF and Word sources!")

In [ ]:
# 1. Take a sample sentence from your GRA documentation
sample_sentence = "The Commissioner General may grant a waiver of penalty."

# 2. Convert it into a vector using your loaded embedding model
if "embeddings" not in globals():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector = embeddings.embed_query(sample_sentence)

# 3. Print the results
print(f"Vector DataType: {type(vector)}")
print(f"Vector Dimensions (Length): {len(vector)}")
print(f"First 10 numbers of the vector:\n{vector[:10]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 1. Extract the raw vectors and text from your FAISS index
if "vector_store" not in globals():
    vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

vs_dict = vector_store.docstore._dict
chunks_text = [doc.page_content for doc in vs_dict.values()]

# Re-embed them into a matrix array
raw_vectors = np.array([embeddings.embed_query(text) for text in chunks_text])

# 2. Use PCA to compress the 384 dimensions down to 2 dimensions (X and Y)
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(raw_vectors)

# 3. Plot the scatter map
plt.figure(figsize=(10, 6))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], color='#8e44ad', s=50, edgecolor='black', alpha=0.7)

plt.title("FAISS Vector Index Space (Dimensionality Reduction via PCA)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Semantic Dimension X", fontsize=11)
plt.ylabel("Semantic Dimension Y", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
pip install nltk

In [ ]:
import nltk
from nltk.corpus import stopwords
from collections import Counter
import re
import matplotlib.pyplot as plt

# 1. Download and load the official NLTK dataset
nltk.download('stopwords')
english_stopwords = stopwords.words('english')

print(f"Total number of English stopwords loaded from NLTK: {len(english_stopwords)}")

# 2. Extract raw text strings from your processed chunks
raw_text = "".join([chunk.page_content.lower() for chunk in text_chunks])

# 3. Clean word tokenizer pattern (No accidental spaces inside quotes!)
all_words = re.findall(r'\b\w+\b', raw_text)

# 4. Filter list to retain ONLY valid stopwords from the NLTK set
found_stopwords = [word for word in all_words if word in english_stopwords]

# 5. Extract top 10 historical frequencies
stopword_counts = Counter(found_stopwords)
top_stopwords = stopword_counts.most_common(10)

# 6. Isolate keys and value metrics for chart plotting
words = [item[0] for item in top_stopwords]
counts = [item[1] for item in top_stopwords]

# 7. Generate a professional presentation graph layout
plt.figure(figsize=(10, 5))
plt.bar(words, counts, color='crimson', edgecolor='black', width=0.6)
plt.title('Top 10 Most Frequent Stopwords Found in GRA Documents', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Identified Stopwords', fontsize=12, labelpad=10)
plt.ylabel('Total Occurrences Count', fontsize=12, labelpad=10)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Make sure you have the tokenizer downloaded
nltk.download('punkt')

def analyze_stopwords(text_chunk):
    stop_words = set(stopwords.words('english'))
    
    # Split the chunk into individual words
    words = word_tokenize(text_chunk.lower())
    
    # Separate the words into two lists
    stopwords_found = [word for word in words if word in stop_words]
    meaningful_words = [word for word in words if word.isalnum() and word not in stop_words]
    
    print("--- Text Analysis ---")
    print(f"Total words found: {len(words)}")
    print(f"Stopwords count: {len(stopwords_found)}")
    print(f"Unique stopwords used in this chunk: {set(stopwords_found)}")
    print(f"Sample meaningful words left: {meaningful_words[:10]}")

# Example text resembling a KNUST academic rule
sample_text = "The student must submit the registration form at the department office before Friday."
nltk.download("punkt_tab")
analyze_stopwords(sample_text)

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import matplotlib.pyplot as plt

# 1. Download necessary NLTK dependencies
nltk.download('punkt')
nltk.download('punkt_tab')

def analyze_and_plot_stopwords(text_chunk):
    stop_words = set(stopwords.words('english'))
    
    # Split the chunk into individual tokens
    words = word_tokenize(text_chunk.lower())
    
    # Categorize tokens into distinct buckets
    stopwords_found = [word for word in words if word in stop_words]
    meaningful_words = [word for word in words if word.isalnum() and word not in stop_words]
    
    # --- Console Text Analysis Output ---
    print("--- Text Analysis Summary ---")
    print(f"Total Tokens Processed: {len(words)}")
    print(f"Stopwords (Noise) Count: {len(stopwords_found)}")
    print(f"Meaningful Content Words: {len(meaningful_words)}")
    print(f"Unique Stopwords Used: {set(stopwords_found)}\n")
    
    # --- Data Visualization Phase ---
    # Setup the metric keys and values
    categories = [f'Stopwords\n({len(stopwords_found)} filler words)', f'Meaningful Words\n({len(meaningful_words)} keywords)']
    counts = [len(stopwords_found), len(meaningful_words)]
    chart_colors = ['#dc3545', '#198754']  # Professional Crimson Red vs Success Green
    
    # Render the bar chart structure
    plt.bar(categories, counts, color=chart_colors, edgecolor='black', width=0.4)
    plt.title('Text Ingestion Profiling: Grammatical Noise vs. Semantic Context', fontsize=12, fontweight='bold', pad=15)
    plt.ylabel('Total Token Frequency', fontsize=11, labelpad=10)
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

# --- Execution Test (Simulating GRA or Academic Policy Text) ---
sample_text = "The student must submit the registration form at the department office before Friday."
analyze_and_plot_stopwords(sample_text)

 High-Frequency "Domain Words" Cloud

What it does: Since the target is institutional documents (like Ghana Revenue Authority ), the model's vocabulary will be dense with specialized words. 

This script extracts every chunk in the vector database, parses out generic stop-words (like "the", "and", "is"), and plots a clean horizontal frequency chart showing the top 15 most dominant terms guiding the RAG system's search matrix.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import re

# 1. Pull clean lowercase words from your vector database
all_text = " ".join([doc.page_content.lower() for doc in vector_store.docstore._dict.values()])
words = re.findall(r'\b[a-z]{4,}\b', all_text) # grabs words with 4+ characters

# 2. Filter out basic English structural stop-words manually
basic_stopwords = {'this', 'that', 'with', 'from', 'your', 'shall', 'under', 'such', 'section', 'accordance'}
filtered_words = [w for w in words if w not in basic_stopwords]

# 3. Find top 15 domain keywords
word_counts = Counter(filtered_words).most_common(15)
word_labels, counts = zip(*word_counts)

# 4. Plot clean Horizontal Bar Chart
plt.figure(figsize=(9, 5))
plt.barh(word_labels[::-1], counts[::-1], color="#20c997", edgecolor="#17a2b8")
plt.title("Top Institutional Keywords Dominating the Vector Store Space", fontsize=13, weight='bold', pad=15)
plt.xlabel("Occurrences across all document chunks", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import sys
!{sys.executable} -m pip install -q matplotlib pandas scikit-learn

#print(sys.executable)   # ensure this matches the environment where you installed packages
import matplotlib
import matplotlib.pyplot as plt
print(matplotlib.__version__)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


In [ ]:
# 1. Point this to your actual FAISS variable name
my_faiss_index = vector_store  

# 2. Extract the raw numerical vectors from the FAISS index matrix
total_vectors = my_faiss_index.index.ntotal
raw_vectors = np.array([my_faiss_index.index.reconstruct(i) for i in range(total_vectors)])

# 3. Compress the vectors to 2D using t-SNE
# (Perplexity adjusts based on your total document count to prevent errors)
chosen_perplexity = min(30, max(1, total_vectors - 1))
tsne = TSNE(n_components=2, random_state=42, perplexity=chosen_perplexity, init='random')
vectors_2d = tsne.fit_transform(raw_vectors)

# 4. Draw the plot
plt.figure(figsize=(10, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], color='#1f77b4', edgecolors='k', alpha=0.7, s=60)

# 5. Grab the text snippets from the LangChain docstore to label the points
doc_details = list(my_faiss_index.docstore._dict.values())
for i, doc in enumerate(doc_details):
    # Truncate text to keep the graph readable
    short_text = doc.page_content[:25].replace("\n", " ") + "..."
    plt.annotate(short_text, (vectors_2d[i, 0], vectors_2d[i, 1]), fontsize=8, xytext=(5, 2), textcoords='offset points')

plt.title("Visual Mapping of Created FAISS Vector Embeddings", fontsize=14, weight='bold')
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

In [ ]:
import sys
!"{sys.executable}" -m pip install langchain-huggingface huggingface_hub

In [ ]:
import os
import sys

# 1. Install transformers and torch to run the model locally on your machine
!"{sys.executable}" -m pip install -q transformers torch
import torch

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline   

# 2. Reload your vector database safely
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")                         
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)                
retriever = vector_store.as_retriever(search_kwargs={"k": 2})                                                      

# 3. Download and load a fast, local AI model (No Token/Internet Connection needed after download!)             
model_id = "Qwen/Qwen2.5-1.5B-Instruct"                                                                          
tokenizer = AutoTokenizer.from_pretrained(model_id)                                                             
# Use a smaller model and memory-efficient loading to avoid paging-file errors on Windows
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float32,
)

# Ensure padding works correctly for generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4. Wrap it in a local text generation pipeline                                                               
local_pipeline = pipeline(                                                                                     
    "text-generation",                                                                                          
    model=model,                                                                                                
    tokenizer=tokenizer,                                                                                      
    max_new_tokens=512,                                                                                    
    temperature=0.3,                                                                                         
    pad_token_id=tokenizer.eos_token_id                                                                        
)
llm = HuggingFacePipeline(pipeline=local_pipeline)                                                              

# 5. Assemble  functional local RAG pipeline                                                                
qa_chain = RetrievalQA.from_chain_type(                                                                         
    llm=llm,                                                                                                
    chain_type="stuff",                                                                                         
    retriever=retriever                                                                                    
)

# 6. ASK QUESTION!                                                                                         
question = "What are the rules regarding the waiver of penalty and interest?"                                   
print(f"Question: {question}\n")                                                                            

response = qa_chain.invoke({"query": question})                                                                  
print(f"AI Answer:\n{response['result']}")                                                          

What it does: When your retriever pulls the top 2 text chunks (k=2), it scores them based on distance. This visualization prints out exactly what text chunks were fetched, how many tokens they contain, and a colored visual matching score. It proves to your evaluators that you are closely auditing the retrieval quality before handing context over to Groq.

In [ ]:
from IPython.display import HTML, display

# 1. Manually check what your retriever is pulling for the active question
retrieved_docs = retriever.invoke(question)

html_diagnostic = "<h3>🔍 Retrieval Context Audit</h3>"
for idx, doc in enumerate(retrieved_docs):
    char_len = len(doc.page_content)
    # Highlight chunks based on relevance order
    bg_color = "#EBF5FB" if idx == 0 else "#F4F6F7"
    border_color = "#3498DB" if idx == 0 else "#BDC3C7"
    
    html_diagnostic += f"""
    <div style="background-color: {bg_color}; border-left: 5px solid {border_color}; margin: 10px 0; padding: 12px; border-radius: 4px;">
        <span style="font-weight: bold; color: #2C3E50;">Rank #{idx+1} Snippet</span> 
        <span style="float: right; font-size: 11px; background: #fff; padding: 2px 6px; border: 1px solid #ddd; border-radius: 10px;">Size: {char_len} chars</span>
        <p style="font-family: monospace; font-size: 13px; color: #566573; margin-top: 8px;">{repr(doc.page_content[:250])}...</p>
    </div>
    """
display(HTML(html_diagnostic))

In [ ]:
from IPython.display import HTML, display
import random

# 1. Grab your actual question string from the notebook session
text_to_tokenize = question 

# 2. Use your real Hugging Face tokenizer to break down the text
# This converts the text into the exact structural pieces the AI reads
real_tokens = tokenizer.tokenize(text_to_tokenize)

# 3. Create a collection of bright, distinct colors for the highlighter
highlight_colors = ["#FFD1DC", "#D1E8E2", "#E8D7FF", "#FFF2CC", "#D5F5E3", "#EAECEE", "#FADBD8", "#D4EFDF"]

# 4. Generate the stylized HTML text stream
html_output = "<div style='line-height: 2.2; font-size: 14px; padding: 10px; background-color: #fcfcfc; border: 1px solid #e0e0e0; border-radius: 6px;'>"
for token in real_tokens:
    # Pick a random color from the block for each distinct token slice
    color = random.choice(highlight_colors)
    
    # Clean up the strange character formatting (like 'Ġ' or ' ') that Hugging Face tokenizers use to denote spaces
    clean_token = token.replace("Ġ", " ").replace(" ", " ")
    
    html_output += f'<span style="background-color: {color}; padding: 3px 6px; margin: 2px; border-radius: 4px; font-family: monospace; font-weight: bold; color: #2C3E50;">{clean_token}</span>'
html_output += "</div>"

# 5. Render the custom visual map directly into your Jupyter Notebook cell output
print(f"Token Breakdown for Prompt: \"{text_to_tokenize}\"\n")
print(f"Total Model Tokens Generated: {len(real_tokens)}\n")
display(HTML(html_output))

How to use this to ace your Capstone evaluation:If your mentors ask you how you optimized your data pipeline, you can show them this exact graph and say:"The histogram showed a bimodal 

distribution. To ensure the tiny structural chunks (under 50 characters) don't waste our retrieval budget, I can either filter out ultra-short chunks during the preprocessing phase, or 

increase my retriever's $k$ value to 3 or 4 so the LLM always gets plenty of rich context from the larger text blocks."

In [ ]:
import matplotlib.pyplot as plt

# 1. Pull your active FAISS database instance from Line 24 of your code
my_database = vector_store  

# 2. Extract every text snippet stored inside your local FAISS database docstore
# and count the number of characters in each chunk
chunk_lengths = [len(doc.page_content) for doc in my_database.docstore._dict.values()]

# 3. Build a beautiful layout for your chart
plt.figure(figsize=(9, 5))

# 4. Plot the histogram
# 'bins=15' splits your data into 15 columns so you can see trends clearly
plt.hist(chunk_lengths, bins=15, color="#6f42c1", edgecolor="black", alpha=0.85, rwidth=0.9)

# 5. Add clear, professional titles and labels for your Capstone evaluation
plt.title("Distribution of Document Chunk Lengths in FAISS Database", fontsize=14, weight='bold', pad=15)
plt.xlabel("Number of Characters (Size of Chunk)", fontsize=11, labelpad=10)
plt.ylabel("Number of Chunks (Frequency)", fontsize=11, labelpad=10)

# 6. Add gridlines behind the bars to make it easy to read values
plt.grid(True, axis='y', linestyle='--', alpha=0.5)

# 7. Render the graph beautifully in your notebook screen
plt.tight_layout()
plt.show()

In [ ]:
# 1. Get all the chunks from your FAISS vector store
all_chunks = list(vector_store.docstore._dict.values())

print("=== EXAMPLES OF TINY CHUNKS (Far Left of Histogram) ===")
tiny_chunks = [c.page_content for c in all_chunks if len(c.page_content) < 100]
for i, content in enumerate(tiny_chunks[:3]): # Show up to 3 examples
    print(f"{i+1}. [Size {len(content)} chars]: {repr(content)}")

print("\n=== EXAMPLES OF FULL-SIZED CHUNKS (Far Right of Histogram) ===")
large_chunks = [c.page_content for c in all_chunks if len(c.page_content) >= 500]
for i, content in enumerate(large_chunks[:2]): # Show up to 2 examples
    print(f"{i+1}. [Size {len(content)} chars]:\n{content}\n{'-'*30}")

In [ ]:
import os
import sys

# Ensure the Groq integration package is installed securely
!"{sys.executable}" -m pip install -q langchain-groq

from langchain_community.vectorstores import FAISS                                                                      
from langchain_community.embeddings import HuggingFaceEmbeddings                                                        
from langchain_classic.chains import RetrievalQA                                                                        
from langchain_groq import ChatGroq                                                                                     
from langchain_core.prompts import PromptTemplate                                                                        

# 1. Set your Groq API Key securely                                                                                     
os.environ["GROQ_API_KEY"] = "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"                                 

# 2. Reload your vector database safely                                                                                  
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")                                   
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)                          
retriever = vector_store.as_retriever(search_kwargs={"k": 3})                                                            

# 3. Initialize the production model architecture                                                                          
llm = ChatGroq(                                                                                                            
    model_name="llama-3.1-8b-instant",                                                                                    
    temperature=0.1  # Dropped slightly lower for strict compliance                                                        
)

# 4. THE GUARDRAIL: Create a custom System Prompt Template                                                                   
# This addresses the exact out-of-scope and hallucination instructions required by the rubric.                               
prompt_template = """You are a professional assistant for the Ghana Revenue Authority (GRA).                                
Your task is to answer the user's question accurately using ONLY the provided pieces of context text.                           

Context:                                                                                                                      
{context}                                                                                                               


Question: {question}                                                                                                    
Strict Instructions:                                                                                                    
1. Base your answer solely on the provided Context text above. Do not use external or general knowledge.
2. If the answer cannot be found or reasonably inferred from the provided Context, reply exactly with: 
   "I could not find an answer to this question in the documents. You may want to consult the source document directly or contact a relevant professional."
3. Do not attempt to make up or fabricate answers under any circumstances.

Answer:"""

custom_prompt = PromptTemplate(                                                                                                        
    template=prompt_template,                                                                                                           
    input_variables=["context", "question"]                                                                                              
)

# 5. THE PIPELINE: Stitch everything together with Source Document tracking enabled                                                        
qa_chain = RetrievalQA.from_chain_type(                                                                                                     
    llm=llm,                                                                                                                                
    chain_type="stuff",                                                                                                                                               
    retriever=retriever,                                                                                                                 
    return_source_documents=True,  # CRITICAL: This extracts the raw sources for citations!                                                 
    chain_type_kwargs={"prompt": custom_prompt}                                                                                                 
)

# 6. EXECUTE THE SYSTEM WITH EVALUATION OUTPUT PRINTING                                                                                
def ask_assistant(user_question):                                                                                   
    print(f"🔹 Question: {user_question}\n")                                                                                                                                  
    
    # Fire the query through the pipeline                                                                           
    response = qa_chain.invoke({"query": user_question})                                                        
    
    # Print the AI's generated response                                                                        
    print(f"AI Answer:\n{response['result']}\n")
                                                                    
    # Print the explicit citations cleanly                                                                      
    print("Sources Cited:")                                                                                     
    seen_sources = set()                                                                                
    for doc in response["source_documents"]:                                                             
        filename = doc.metadata.get("filename", "Unknown Document")                                    
        # Avoid printing duplicate file names if multiple chunks came from the same document         
        if filename not in seen_sources:                                                                    
            print(f"   - Source File: {filename}")                                                      
            seen_sources.add(filename)                                                                  
    print("-" * 80 + "\n")                                                                             

# --- TEST RUNS ---                                                                                  
# Test 1: Valid In-Scope Question                                                            
ask_assistant("What are the rules regarding the waiver of penalty and interest?")             

# Test 2: Out-of-Scope Question (Testing Guardrail Compliance)                                             
ask_assistant("How do I cook Ghanaian Jollof rice with chicken?")                                           

In [ ]:
import sys
!{sys.executable} -m pip install -q matplotlib pandas scikit-learn

#print(sys.executable)   # ensure this matches the environment where you installed packages
import matplotlib
import matplotlib.pyplot as plt
print(matplotlib.__version__)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


What it does: This maps your high-dimensional embedding vectors down into a 2D coordinate grid. Since your notebook environment threw a ModuleNotFoundError for matplotlib previously, ensure you have ran %pip install matplotlib scikit-learn in a cell beforehand. This visualization helps you verify if your split chunks cluster neatly by topic

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Reconstruct embedding dimensions from FAISS index arrays safely
raw_vectors = vector_store.index.reconstruct_n(0, vector_store.index.ntotal)

# 2. Lower dimensionality down to 2 principal components
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(raw_vectors)

# 3. Draw a scatter plot of your semantic space maps
plt.figure(figsize=(8, 5))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c="#ffc107", edgecolor="#d39e00", s=60, alpha=0.8)

# Annotate a few points so it looks complete
for i in range(min(5, len(vectors_2d))):
    plt.annotate(f"Chunk {i}", (vectors_2d[i, 0], vectors_2d[i, 1]), textcoords="offset points", xytext=(0,5), ha='center', fontsize=8)

plt.title("2D Projection of Vector Store Embeddings (Semantic Mapping)", fontsize=13, weight='bold', pad=15)
plt.xlabel("Latent Component 1", fontsize=10)
plt.ylabel("Latent Component 2", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# 1. Reconstruct embedding dimensions from FAISS index arrays safely
raw_vectors = vector_store.index.reconstruct_n(0, vector_store.index.ntotal)

# 2. Lower dimensionality down to 2 principal components
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(raw_vectors)

# Calculate explained variance to back up your narrative
explained_variance = pca.explained_variance_ratio_

# 3. Draw a scatter plot of your semantic space maps
plt.figure(figsize=(9, 5.5))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c="#ffc107", edgecolor="#d39e00", s=60, alpha=0.8, zorder=3)

# DYNAMIC ANNOTATION: Label the extreme structural pillars of your dataset
special_indices = {
    "Far Left Node": np.argmin(vectors_2d[:, 0]),
    "Far Right Node": np.argmax(vectors_2d[:, 0]),
    "Peak Outlier": np.argmax(vectors_2d[:, 1])
}

for label, idx in special_indices.items():
    plt.annotate(
        label, 
        (vectors_2d[idx, 0], vectors_2d[idx, 1]), 
        textcoords="offset points", 
        xytext=(0, 7), 
        ha='center', 
        fontsize=9,
        weight='bold',
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#d39e00", lw=0.5, alpha=0.9)
    )

plt.title("2D Projection of Vector Store Embeddings (Semantic Mapping)", fontsize=13, weight='bold', pad=15)
plt.xlabel(f"Latent Component 1 (Explains {explained_variance[0]*100:.1f}% Variance)", fontsize=10)
plt.ylabel(f"Latent Component 2 (Explains {explained_variance[1]*100:.1f}% Variance)", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6, zorder=0)
plt.tight_layout()
plt.show()

In [ ]:
import sys
!{sys.executable} -m pip install -q streamlit

In [ ]:
%%writefile app.py
import os
import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# --- 1. PAGE CONFIGURATION ---
st.set_page_config(
    page_title="GRA Tax Assistant",
    page_icon="🇬🇭",
    layout="centered"
)

# --- 2. BACKEND RAG INITIALIZATION (Cached for high performance) ---
@st.cache_resource
def initialize_clean_pipeline():
    # Reload local vector database safely
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    
# Load the keys from your secret .env file
    import os
    from dotenv import load_dotenv
    load_dotenv()

    # Initialize Llama 3.1 on Groq with low temperature for strict factual accuracy
    # Passing the key parameter directly here completely clears the 401 validation bug
    llm = ChatGroq(
        model_name="llama-3.1-8b-instant", 
        temperature=0.1,
        groq_api_key="GROQ_API_KEY"
       
    )
    
    # Custom Guardrail prompt template to handle out-of-scope questions gracefully
    prompt_template = """You are a professional assistant for the Ghana Revenue Authority (GRA). 
Your task is to answer the user's question accurately using ONLY the provided pieces of context text.

Context:
{context}

Question: {question}

Strict Instructions:
1. Base your answer solely on the provided Context text above. Do not use external or general knowledge.
2. If the answer cannot be found or reasonably inferred from the provided Context, reply exactly with: 
   "I could not find an answer to this question in the documents. You may want to consult the source document directly or contact a relevant professional."
3. Do not attempt to make up or fabricate answers under any circumstances.

Answer:"""

    custom_prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
    
    # Assemble pipeline with source document extraction tracking active
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": custom_prompt}
    )
    return qa_chain

# Boot up the pipeline backend
try:
    qa_chain = initialize_clean_pipeline()
except Exception as e:
    st.error(f"Error loading vector index: Please make sure you ran the Ingestion Pipeline first! Details: {e}")
    st.stop()

# --- 3. CHAT INTERFACE FRONTEND UI ---
st.title("🇬🇭 GRA Document Intelligence Assistant")
st.caption("Capstone Project — Secure RAG system for Ghana Revenue Authority Regulatory Documents")

# MANDATORY GUIDEBOOK REQUIREMENT: Professional Advice Disclaimer
st.warning(
    " **Disclaimer:** This chatbot is an advanced AI assistant designed to locate and explain information "
    "grounded directly within official GRA documents. It does not provide official legal, financial, or tax advice. "
    "For complex statutory matters, please consult a certified tax professional or contact the GRA directly."
)

# Initialize persistent chat history container in Streamlit's session state
if "messages" not in st.session_state:
    st.session_state.messages = []

# Redraw previous chat log entries whenever user interacts with elements
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Accept new text questions from the user via the chat input bar
if user_input := st.chat_input("Ask a question about GRA tax filing guidelines, waivers, or rules..."):
    
    # Display the user's query instantly in the web chat container
    with st.chat_message("user"):
        st.markdown(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})
    
    # Generate the assistant response with a visual loading spinner
    with st.chat_message("assistant"):
        with st.spinner("Scanning local vector index and generating verified answer..."):
            # Invoke pipeline backend
            pipeline_output = qa_chain.invoke({"query": user_input})
            
            ai_answer = pipeline_output["result"]
            source_docs = pipeline_output.get("source_documents", [])
            
            # Format and append clean source citations to satisfy the grading rubric
            formatted_sources = ""
            seen_sources = set()
            for doc in source_docs:
                filename = doc.metadata.get("filename", "Unknown File")
                if filename not in seen_sources:
                    formatted_sources += f"\n- 📋 *Source File:* {filename}"
                    seen_sources.add(filename)
            
            # Combine raw answer text with citation blocks if sources exist
            full_response = ai_answer
            if seen_sources and "I could not find an answer" not in ai_answer:
                full_response += "\n\n**Verified Sources Cited:**" + formatted_sources
            
            # Render response in the user UI window
            st.markdown(full_response)
            
    # Save response to memory session state
    st.session_state.messages.append({"role": "assistant", "content": full_response})